# Kelly Fine-Tune v1 — Kaggle Edition

**Plataforma:** Kaggle (T4 x2 GPU grátis, 30h/semana, sem rate limit pesado do Colab).

Salva o modelo em **2 lugares** (Drive Lissy removido aqui):
1. **HuggingFace Hub** (`wellyton5/kelly-llama3.1-8b`) — primário, permanente
2. **judith-arm via SSH** — deploy direto pra Ollama

LoRA adapters checkpoint salvo no HF logo após treino — segurança extra.

---

## ANTES DE RODAR

### 1. Habilitar GPU + Internet
- Settings (canto direito) → **Accelerator** → **GPU T4 x2** (recomendado, 2x mais rápido) ou **GPU P100**
- Settings → **Internet** → **On** (precisa pra HuggingFace, GitHub, judith-arm)

### 2. Adicionar Secrets
- Add-ons → Secrets:
  - `HF_TOKEN` = seu token Write do HuggingFace
  - `JUDITH_ARM_SSH_KEY` = conteúdo da chave SSH (cat ~/.ssh/judith-arm-key na Phoenix)

### 3. Run All
Click "Save Version" → "Save & Run All (Commit)" ou Run all manual.
Aguarde ~25-35 min total (Kaggle T4 x2 é mais rápido que Colab).


In [ ]:
# ============================================================
# Cell 1: Setup — secrets, paths, configs (Kaggle version)
# ============================================================
import os, base64
from kaggle_secrets import UserSecretsClient

us = UserSecretsClient()
HF_TOKEN = us.get_secret('HF_TOKEN')
SSH_KEY_B64 = us.get_secret('JUDITH_ARM_SSH_KEY_B64')
JUDITH_ARM_SSH_KEY = base64.b64decode(SSH_KEY_B64).decode('utf-8')

assert HF_TOKEN, 'HF_TOKEN nao encontrado nos Kaggle Secrets'
assert JUDITH_ARM_SSH_KEY, 'JUDITH_ARM_SSH_KEY_B64 nao encontrado/invalido'

# Config
HF_REPO = 'wellyton5/kelly-llama3.1-8b'
JUDITH_ARM_IP = '129.146.230.48'
JUDITH_ARM_USER = 'opc'
MODEL_NAME = 'kelly-llama3.1-8b'
DATASET_URL = 'https://prospecdesign.com.br/static/kelly_training_data.jsonl'

# Setup SSH key
SSH_KEY_PATH = '/root/.ssh/judith-arm-key'
os.makedirs('/root/.ssh', exist_ok=True)
with open(SSH_KEY_PATH, 'w') as f:
    f.write(JUDITH_ARM_SSH_KEY.strip() + chr(10))
os.chmod(SSH_KEY_PATH, 0o600)

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

print('OK: secrets carregados')
print(f'   Modelo:          {MODEL_NAME}')
print(f'   HF repo destino: {HF_REPO}')
print(f'   judith-arm:      {JUDITH_ARM_USER}@{JUDITH_ARM_IP}')
print(f'   Dataset:         {DATASET_URL}')


In [ ]:
# ============================================================
# Cell 2: Install Unsloth + verify GPU
# Versao DEFINITIVA — ambos pacotes do GitHub main pra evitar mismatch
# ============================================================
%%capture
!pip install -q --upgrade pip
# Instalar dependencias do PyPI primeiro
!pip install -q bitsandbytes accelerate xformers peft trl datasets transformers
# DEPOIS sobrescrever unsloth + unsloth_zoo do GitHub (sincronizados)
!pip install -q --no-deps --upgrade git+https://github.com/unslothai/unsloth-zoo.git
!pip install -q --no-deps --upgrade git+https://github.com/unslothai/unsloth.git
!pip install -q huggingface_hub


In [ ]:
# ============================================================
# Cell 3: Verify GPU + download dataset
# ============================================================
import torch
import urllib.request

assert torch.cuda.is_available(), 'GPU nao disponivel - mude Runtime para T4'
gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name}  |  VRAM: {vram:.1f} GB')

# Download dataset
urllib.request.urlretrieve(DATASET_URL, 'kelly_training_data.jsonl')
with open('kelly_training_data.jsonl') as f:
    n_examples = sum(1 for _ in f)
print(f'Dataset: {n_examples} exemplos baixados de {DATASET_URL}')

In [ ]:
# ============================================================
# Cell 4: Load Llama 3.1 8B base + LoRA config
# ============================================================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
)
model.print_trainable_parameters()

In [ ]:
# ============================================================
# Cell 5: Format dataset com chat template Llama 3.1
# ============================================================
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template='llama-3.1')
full_dataset = load_dataset('json', data_files='kelly_training_data.jsonl', split='train')
split = full_dataset.train_test_split(test_size=0.1, seed=3407)
dataset, val_dataset = split['train'], split['test']

def fmt(examples):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in examples['conversations']]
    return {'text': texts}

dataset = dataset.map(fmt, batched=True)
val_dataset = val_dataset.map(fmt, batched=True)
print(f'Train: {len(dataset)}  |  Val: {len(val_dataset)}')

In [ ]:
# ============================================================
# Cell 6: Train (5 epochs, ~30-60 min na T4)
# ============================================================
from unsloth import UnslothTrainer, UnslothTrainingArguments, is_bfloat16_supported

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=val_dataset,
    dataset_text_field='text',
    max_seq_length=4096,
    packing=False,
    args=UnslothTrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=3407,
        output_dir='outputs',
    ),
)
stats = trainer.train()
print(f'\nDONE! Train loss: {stats.metrics["train_loss"]:.4f}')

In [ ]:
# ============================================================
# Cell 7: CHECKPOINT — LoRA adapters → HuggingFace (~80 MB, fast)
# Roda IMEDIATAMENTE após treino. Se export GGUF falhar depois,
# voce pode re-exportar de qualquer maquina sem retreinar.
# ============================================================
print('Salvando LoRA adapters localmente...')
ADAPTERS_DIR = 'kelly-lora-adapters'
model.save_pretrained(ADAPTERS_DIR)
tokenizer.save_pretrained(ADAPTERS_DIR)

import os
adapter_size_mb = sum(os.path.getsize(os.path.join(ADAPTERS_DIR, f))
                      for f in os.listdir(ADAPTERS_DIR)
                      if os.path.isfile(os.path.join(ADAPTERS_DIR, f))) / 1e6
print(f'  Adapters: {adapter_size_mb:.1f} MB em {ADAPTERS_DIR}/')

# Push adapters pro HuggingFace (subpasta lora-adapters/)
from huggingface_hub import HfApi, create_repo
api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO, repo_type='model', private=False, exist_ok=True, token=HF_TOKEN)

print(f'\\nUploading adapters para {HF_REPO}/lora-adapters/ ...')
try:
    api.upload_folder(
        folder_path=ADAPTERS_DIR,
        path_in_repo='lora-adapters',
        repo_id=HF_REPO,
        repo_type='model',
        token=HF_TOKEN,
    )
    print(f'\\n[CHECKPOINT] OK Adapters seguros em https://huggingface.co/{HF_REPO}/tree/main/lora-adapters')
except Exception as e:
    print(f'\\n[CHECKPOINT] AVISO: upload de adapters falhou: {e}')
    print('             Continuando — adapters ainda estao no disco do Colab.')

In [ ]:
# ============================================================
# Cell 8: Export GGUF Q8_0 (15-25 min)
# ============================================================
print('Exportando GGUF Q8_0...')
print('  [1/3] Merging weights into 16bit...')
print('  [2/3] HF -> GGUF F16...')
print('  [3/3] F16 -> Q8_0...')
print('Total esperado: ~20 min na T4.')
print()

import glob, os
GGUF_PATH = None
GGUF_SIZE_GB = 0
try:
    model.save_pretrained_gguf(MODEL_NAME, tokenizer, quantization_method='q8_0')
    candidates = glob.glob(f'{MODEL_NAME}_gguf/*.gguf') + glob.glob(f'{MODEL_NAME}/*.gguf')
    assert candidates, 'Nenhum GGUF gerado!'
    GGUF_PATH = candidates[0]
    GGUF_SIZE_GB = os.path.getsize(GGUF_PATH) / 1e9
    print(f'\\n[GGUF] OK {GGUF_PATH} ({GGUF_SIZE_GB:.1f} GB)')
except Exception as e:
    print(f'\\n[GGUF] FALHOU: {e}')
    print('Adapters ja estao seguros no HF (cell 7).')
    print('Pode reexportar GGUF depois de qualquer maquina com:')
    print('  pip install unsloth && from unsloth import FastLanguageModel')
    print(f'  model = FastLanguageModel.from_pretrained(\"{HF_REPO}/lora-adapters\")')
    print(f'  model.save_pretrained_gguf(\"{MODEL_NAME}\", tokenizer, \"q8_0\")')
    raise

In [ ]:
# ============================================================
# Cell 9: SAVE 1/3 — HuggingFace Hub GGUF (primario, permanente)
# ============================================================
SAVE_RESULTS = {'hf': False, 'drive': False, 'judith_arm': False}
HF_DOWNLOAD_URL = None

try:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)

    print(f'Uploading GGUF {GGUF_SIZE_GB:.1f} GB para HuggingFace... (10-15 min)')
    api.upload_file(
        path_or_fileobj=GGUF_PATH,
        path_in_repo=os.path.basename(GGUF_PATH),
        repo_id=HF_REPO,
        repo_type='model',
        token=HF_TOKEN,
    )

    # README com metadata do treino
    readme = f'''---
license: llama3.1
base_model: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
tags: [unsloth, llama, llama-3.1, gguf, kelly, prospec]
---

# Kelly Llama 3.1 8B

Fine-tune Unsloth + LoRA do Meta Llama 3.1 8B Instruct.

Kelly atua como **fallback de raciocinio local** para os agentes Eva e Judith
que rodam na infra da Prospec & Design LLC. Quando os providers de LLM
externos (Gemini, Mistral, Groq, etc.) falham por rate limit ou quota
esgotada, Kelly assume o trabalho garantindo que o sistema nunca pare.

## Training details
- Base: `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`
- LoRA: r=16, alpha=16, dropout=0, target=q/k/v/o/gate/up/down_proj
- Trainable params: 41,943,040 (0.52%)
- Dataset: 576 examples (90% train / 10% val)
- Epochs: 5, batch=2x4, lr=2e-4 cosine, optim=adamw_8bit
- Train loss final: {stats.metrics["train_loss"]:.4f}
- Quantization: Q8_0 GGUF ({GGUF_SIZE_GB:.1f} GB)

## Files
- `{os.path.basename(GGUF_PATH)}` — GGUF Q8_0 ready for Ollama
- `lora-adapters/` — LoRA adapters checkpoint (~80 MB) for re-export

## Use with Ollama
```bash
wget https://huggingface.co/{HF_REPO}/resolve/main/{os.path.basename(GGUF_PATH)}
ollama create kelly -f Modelfile
```
'''
    with open('README.md', 'w') as f:
        f.write(readme)
    api.upload_file(path_or_fileobj='README.md', path_in_repo='README.md',
                    repo_id=HF_REPO, repo_type='model', token=HF_TOKEN)

    HF_DOWNLOAD_URL = f'https://huggingface.co/{HF_REPO}/resolve/main/{os.path.basename(GGUF_PATH)}'
    SAVE_RESULTS['hf'] = True
    print(f'\n[1/3] OK HuggingFace: {HF_DOWNLOAD_URL}')
except Exception as e:
    print(f'\n[1/3] FALHOU HF: {e}')
    print('Continuando para tentar Drive e judith-arm...')

In [ ]:
# ============================================================
# Cell 11: SAVE 3/3 — Trigger deploy na judith-arm via SSH
# Estrategia: judith-arm baixa o GGUF do HF (mais rapido e robusto que SCP gigante)
# ============================================================
import subprocess

if not SAVE_RESULTS['hf'] or not HF_DOWNLOAD_URL:
    print('[3/3] PULADO judith-arm: sem URL no HF para baixar.')
    print('      (Cell 9 falhou — judith-arm nao tem fonte pra puxar)')
else:
    deploy_cmd = f'''set -e
mkdir -p ~/kelly-model
cd ~/kelly-model

echo "[judith-arm] Baixando Kelly GGUF do HuggingFace ({GGUF_SIZE_GB:.1f} GB)..."
wget -q --show-progress -c -O {os.path.basename(GGUF_PATH)} {HF_DOWNLOAD_URL}

echo "[judith-arm] Verificando tamanho..."
ls -lh {os.path.basename(GGUF_PATH)}

echo "[judith-arm] DONE — modelo em ~/kelly-model/{os.path.basename(GGUF_PATH)}"
echo "[judith-arm] Pra ativar no Ollama, rode:"
echo "[judith-arm]    curl -fsSL https://prospecdesign.com.br/static/deploy_kelly.sh | bash"
'''

    ssh_args = [
        'ssh', '-i', SSH_KEY_PATH,
        '-o', 'StrictHostKeyChecking=no',
        '-o', 'UserKnownHostsFile=/dev/null',
        '-o', 'ConnectTimeout=15',
        f'{JUDITH_ARM_USER}@{JUDITH_ARM_IP}',
        deploy_cmd,
    ]

    print('Disparando download do Kelly via SSH na judith-arm...')
    try:
        result = subprocess.run(ssh_args, capture_output=True, text=True, timeout=1200)
        print(result.stdout)
        if result.returncode != 0:
            print(f'STDERR:\n{result.stderr}')
            raise RuntimeError(f'SSH retornou rc={result.returncode}')
        SAVE_RESULTS['judith_arm'] = True
        print(f'\n[3/3] OK judith-arm: ~/kelly-model/{os.path.basename(GGUF_PATH)}')
    except subprocess.TimeoutExpired:
        print('\n[3/3] FALHOU judith-arm: timeout (>20 min). Pode rodar manualmente depois:')
        print(f'      ssh {JUDITH_ARM_USER}@{JUDITH_ARM_IP}')
        print(f'      curl -fsSL https://prospecdesign.com.br/static/deploy_kelly.sh | bash')
    except Exception as e:
        print(f'\n[3/3] FALHOU judith-arm: {e}')
        print(f'      Voce pode rodar manualmente depois:')
        print(f'      ssh {JUDITH_ARM_USER}@{JUDITH_ARM_IP}')
        print(f'      curl -fsSL https://prospecdesign.com.br/static/deploy_kelly.sh | bash')

In [ ]:
# ============================================================
# Cell 12: Smoke test (POR ULTIMO)
# Notebook v2 quebrava aqui ANTES do save. Aqui o save ja terminou
# entao se quebrar, nao perde nada — modelo ja esta nos 3 lugares.
# ============================================================
print('Smoke test (modelo ja foi salvo, falha aqui nao impacta nada)')
print()

try:
    FastLanguageModel.for_inference(model)
    system_msg = 'You are Kelly, secretary and sales agent at Prospec & Design LLC, a professional cleaning company in Austin, TX.'
    msgs = [
        {'role':'system','content':system_msg},
        {'role':'user','content':'Quais servicos voces oferecem?'},
    ]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, use_cache=True, do_sample=True)
    response = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
    print('Resposta da Kelly:')
    print(response[:500])
except Exception as e:
    # Bug classico shape mismatch [1, 32, 1, 128] vs [1, 32, N, 128] do Unsloth+Llama3.1+Transformers5.3
    # Nao impede o modelo de funcionar — eh apenas no .generate() do Colab.
    # O GGUF exportado e o Ollama na judith-arm funcionam normalmente.
    print(f'Smoke test no Colab falhou: {e}')
    print()
    print('Isso nao significa que a Kelly esta ruim — eh um bug conhecido')
    print('do Unsloth+Llama3.1+Transformers5.3 no .generate() pos-LoRA.')
    print('O GGUF exportado funciona normal no Ollama da judith-arm.')
    print()
    print('Pra testar, rode na judith-arm:')
    print(f'  ssh {JUDITH_ARM_USER}@{JUDITH_ARM_IP}')
    print('  ollama run kelly')

In [ ]:
# ============================================================
# Cell final: Resumo
# ============================================================
print("=" * 60)
print("  KELLY FINE-TUNE v1 — KAGGLE — DONE")
print("=" * 60)
print(f"Modelo:        {MODEL_NAME}")
print(f"Tamanho GGUF:  {GGUF_SIZE_GB:.1f} GB")
print(f"Train loss:    {stats.metrics['train_loss']:.4f}")
print()
print("Salvamentos:")
hf_status   = "OK" if SAVE_RESULTS["hf"] else "FALHOU"
arm_status  = "OK" if SAVE_RESULTS["judith_arm"] else "FALHOU"
print(f"  [1/2] HuggingFace:  {hf_status:8s}  {HF_DOWNLOAD_URL or 'sem URL'}")
print(f"  [2/2] judith-arm:   {arm_status:8s}  ~/kelly-model/")
print()
n_ok = sum([SAVE_RESULTS["hf"], SAVE_RESULTS["judith_arm"]])
if n_ok == 2:
    print("TUDO OK — Kelly em 2 lugares (HF + judith-arm).")
elif n_ok >= 1:
    print(f"AVISO: {n_ok}/2 saves OK.")
else:
    print("CRITICO: nenhum save funcionou.")
print()
print("Proximo passo:")
print(f"  ssh {JUDITH_ARM_USER}@{JUDITH_ARM_IP}")
print("  curl -fsSL https://prospecdesign.com.br/static/deploy_kelly.sh | bash")
